## Best Model
Fine-tunes HateBERT for PCL binary classification using:
- Weighted cross-entropy loss to address class imbalance
- Keyword prepended to input text for community-aware context
- Increased dropout (0.3) on classification head to reduce overfitting
  

### Imports

In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report
import ast
import os

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


### Config

In [13]:
DATA_DIR       = '../datasets'
TSV_PATH       = os.path.join(DATA_DIR, 'dontpatronizeme_pcl.tsv')
TRAIN_PATH     = os.path.join(DATA_DIR, 'train_semeval_parids-labels.csv')
DEV_PATH       = os.path.join(DATA_DIR, 'dev_semeval_parids-labels.csv')

MODEL_NAME     = 'GroNLP/hateBERT'
MAX_LEN        = 128     # covers >95% of paragraphs based on EDA (mean=48, median=42)
BATCH_SIZE     = 32
EPOCHS         = 10
LEARNING_RATE  = 1e-5
WARMUP_RATIO   = 0.1     # 10% of steps used for warmup

LOAD_MODEL =  True #set to false to train from scratch
CHECKPOINT = 'best_model.pt'


### Load data 

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PCLDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'      : encoding['input_ids'].squeeze(),
            'attention_mask' : encoding['attention_mask'].squeeze(),
            'label'          : torch.tensor(self.labels[idx], dtype=torch.long)
        }


In [6]:
df = pd.read_csv(
    TSV_PATH,
    sep='\t',
    header=None,
    names=['par_id', 'art_id', 'keyword', 'country', 'text', 'label'],
    quoting=3,
    engine='python',
    skiprows=4
)

# Binary label: {0,1} -> 0 (No PCL), {2,3,4} -> 1 (PCL)
df['binary_label'] = df['label'].apply(lambda x: 0 if x in [0, 1] else 1)

# Load split files — we only need par_id from these
train_ids = pd.read_csv(TRAIN_PATH)['par_id'].astype(int).tolist()
dev_ids   = pd.read_csv(DEV_PATH)['par_id'].astype(int).tolist()

train_df = df[df['par_id'].isin(train_ids)].reset_index(drop=True)
dev_df   = df[df['par_id'].isin(dev_ids)].reset_index(drop=True)

#Make sure par_id from tsv is also int
df['par_id'] = df['par_id'].astype(int)

# Prepend the keyword to each text
train_texts_keyword = [
    f"{kw} {txt}" for kw, txt in zip(train_df['keyword'], train_df['text'])
]

dev_texts_keyword = [
    f"{kw} {txt}" for kw, txt in zip(dev_df['keyword'], dev_df['text'])
]

train_dataset_keyword = PCLDataset(
    train_texts_keyword,
    train_df['binary_label'].tolist(),
    tokenizer,
    MAX_LEN
)

dev_dataset_keyword = PCLDataset(
    dev_texts_keyword,
    dev_df['binary_label'].tolist(),
    tokenizer,
    MAX_LEN
)

train_loader = DataLoader(
    train_dataset_keyword,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED)
)

dev_loader = DataLoader(
    dev_dataset_keyword,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(f'Train batches (keyword): {len(train_loader)} | Dev batches (keyword): {len(dev_loader)}')
total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)
print(f'Total steps: {total_steps} | Warmup steps: {warmup_steps}')

Train batches (keyword): 262 | Dev batches (keyword): 66
Total steps: 2620 | Warmup steps: 262


### Calculate class weights

In [7]:
# Weight each class inversely proportional to its frequency
n_samples  = len(train_df)
n_pcl      = train_df['binary_label'].sum()
n_no_pcl   = n_samples - n_pcl

weight_no_pcl = n_samples / (2 * n_no_pcl)
weight_pcl    = n_samples / (2 * n_pcl)

class_weights = torch.tensor([weight_no_pcl, weight_pcl], dtype=torch.float).to(device)
print(f'Class weights -> No PCL: {weight_no_pcl:.4f} | PCL: {weight_pcl:.4f}')

Class weights -> No PCL: 0.5524 | PCL: 5.2739


### Training Loop

In [8]:
def train_epoch(model, loader, optimiser, scheduler, loss_fn, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimiser.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = loss_fn(outputs.logits, labels)
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimiser.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1       = f1_score(all_labels, all_preds, pos_label=1)
    return avg_loss, f1


def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs    = model(input_ids=input_ids, attention_mask=attention_mask)
            loss       = loss_fn(outputs.logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1       = f1_score(all_labels, all_preds, pos_label=1)
    return avg_loss, f1, all_preds, all_labels

### Model and training

In [14]:
if LOAD_MODEL:
    print(f'Loading checkpoint from {CHECKPOINT}...')
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
    model = model.to(device)
    print('Checkpoint loaded — skipping training.')
else: 
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    model.classifier.dropout = nn.Dropout(p=0.3)
    model = model.to(device)
    
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    
    optimiser = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(optimiser, warmup_steps, total_steps)
    
    best_f1 = 0
    
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_f1 = train_epoch(model, train_loader, optimiser, scheduler, loss_fn, device)
        dev_loss, dev_f1, _, _ = evaluate(model, dev_loader, loss_fn, device)
    
        print(f'Epoch {epoch}/{EPOCHS}')
        print(f'  Train -> Loss: {train_loss:.4f} | F1: {train_f1:.4f}')
        print(f'  Dev   -> Loss: {dev_loss:.4f}   | F1: {dev_f1:.4f}')
    
        # Save best model based on dev F1
        if dev_f1 > best_f1:
            best_f1 = dev_f1
            torch.save(model.state_dict(), 'best_model.pt')
            print(f'  >> New best model saved (F1: {best_f1:.4f})')
    
    print(f'\nBest dev F1 (with weighted loss): {best_f1:.4f}')

model.load_state_dict(torch.load('best_model.pt', map_location=device))
_, dev_f1, preds, labels = evaluate(model, dev_loader, loss_fn, device)

print('=== Final Dev Set Results ===')
print(f'F1 (PCL class): {dev_f1:.4f}')
print()
print(classification_report(labels, preds, target_names=['No PCL', 'PCL']))

Loading checkpoint from best_model.pt...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Checkpoint loaded — skipping training.
=== Final Dev Set Results ===
F1 (PCL class): 0.5714

              precision    recall  f1-score   support

      No PCL       0.96      0.95      0.95      1895
         PCL       0.57      0.57      0.57       199

    accuracy                           0.92      2094
   macro avg       0.76      0.76      0.76      2094
weighted avg       0.92      0.92      0.92      2094



## Generating Predictions

In [15]:
def generate_predictions(model, df, tokenizer, max_len, batch_size, device, output_file):
    texts   = [f"{kw} {txt}" for kw, txt in zip(df['keyword'], df['text'])]
    dataset = PCLDataset(texts, [0]*len(texts), tokenizer, max_len)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
            preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())

    with open(output_file, 'w') as f:
        for p in preds:
            f.write(f'{p}\n')

    print(f'Saved {output_file} | {len(preds)} predictions | PCL: {sum(preds)} ({100*sum(preds)/len(preds):.1f}%)')


In [16]:
generate_predictions(model, dev_df, tokenizer, MAX_LEN, BATCH_SIZE, device, 'dev.txt')

TEST_PATH = os.path.join(DATA_DIR, 'task4_test.tsv')
test_df = pd.read_csv(
    TEST_PATH,
    sep='\t',
    header=None,
    names=['id', 'art_id', 'keyword', 'country', 'text'],
    quoting=3,
    engine='python'
)

generate_predictions(model, test_df, tokenizer, MAX_LEN, BATCH_SIZE, device, 'test.txt')

Saved dev.txt | 2094 predictions | PCL: 200 (9.6%)
Saved test.txt | 3832 predictions | PCL: 335 (8.7%)
